# Tahap 5 — Model Evaluation

Mengukur dan menganalisis performa retrieval dan prediksi.

**Metrik untuk label_pasal**: Accuracy, Precision, Recall, F1-Score

**Metrik untuk vonis_bulan**: MAE (Mean Absolute Error) dalam satuan bulan

**Input**:
- `data/eval/retrieval_metrics.csv`
- `data/results/predictions.csv`

**Output**:
- `data/eval/prediction_metrics.csv`
- `data/eval/evaluation_chart.png`

## 5.1 Import Library

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, precision_score, recall_score, mean_absolute_error
)

warnings.filterwarnings("ignore")
%matplotlib inline

## 5.2 Load Data

In [ ]:
EVAL_DIR   = Path("../data/eval")
RESULT_DIR = Path("../data/results")

df_ret  = pd.read_csv(EVAL_DIR / "retrieval_metrics.csv")
df_pred = pd.read_csv(RESULT_DIR / "predictions.csv")

print("Retrieval Metrics (dari Tahap 3):")
display(df_ret)

print(f"\nTotal query evaluasi: {len(df_pred)}")
print(f"Kolom: {list(df_pred.columns)}")

## 5.3 Evaluasi Prediksi label_pasal

In [ ]:
def compute_pasal_metrics(y_true, y_pred, model_name):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    print(f"\n{'='*55}")
    print(f"  {model_name} | label_pasal")
    print(f"{'='*55}")
    print(classification_report(y_true, y_pred, zero_division=0))
    return {
        "model": model_name, "label": "label_pasal",
        "accuracy": round(acc, 4), "precision": round(prec, 4),
        "recall": round(rec, 4), "f1_score": round(f1, 4),
        "n_samples": len(y_true),
    }


gt_pasal = df_pred["ground_truth_pasal"].tolist()

pred_cols = {
    "TF-IDF + MajorityVote": "tfidf_mv_pasal",
    "TF-IDF + WeightedVote": "tfidf_wv_pasal",
    "IndoBERT + MajorityVote": "bert_mv_pasal",
    "IndoBERT + WeightedVote": "bert_wv_pasal",
}

pred_metrics = []
for model_name, col in pred_cols.items():
    if col in df_pred.columns and df_pred[col].notna().any():
        y_pred = df_pred[col].fillna("tidak_diketahui").tolist()
        m = compute_pasal_metrics(gt_pasal, y_pred, model_name)
        pred_metrics.append(m)

df_pred_metrics = pd.DataFrame(pred_metrics)
df_pred_metrics.to_csv(EVAL_DIR / "prediction_metrics.csv", index=False)
print("\nPREDICTION METRICS SUMMARY:")
display(df_pred_metrics)

## 5.4 Evaluasi Prediksi vonis_bulan (MAE)

MAE (Mean Absolute Error) mengukur rata-rata selisih absolut antara vonis prediksi dan vonis sebenarnya dalam satuan bulan.

In [ ]:
gt_vonis = df_pred["ground_truth_vonis"].tolist()

vonis_cols = {
    "TF-IDF + MajorityVote": "tfidf_mv_vonis",
    "TF-IDF + WeightedVote": "tfidf_wv_vonis",
    "IndoBERT + MajorityVote": "bert_mv_vonis",
    "IndoBERT + WeightedVote": "bert_wv_vonis",
}

print("\nMAE Prediksi vonis_bulan:")
print(f"  Ground truth vonis: {gt_vonis}")
vonis_mae = []
for model_name, col in vonis_cols.items():
    if col in df_pred.columns and df_pred[col].notna().any():
        y_pred = df_pred[col].fillna(0).tolist()
        valid_idx = [i for i, v in enumerate(gt_vonis) if v != 0]
        if valid_idx:
            yt = [gt_vonis[i] for i in valid_idx]
            yp = [y_pred[i] for i in valid_idx]
            mae = mean_absolute_error(yt, yp)
            print(f"  {model_name}: MAE = {mae:.2f} bulan")
            vonis_mae.append({"model": model_name, "mae_bulan": round(mae, 2)})

df_vonis_mae = pd.DataFrame(vonis_mae)
display(df_vonis_mae)

## 5.5 Visualisasi Bar Chart — Accuracy & F1-Score

In [ ]:
df_r = df_ret[df_ret["label"] == "label_pasal"].copy()
df_p = df_pred_metrics[df_pred_metrics["label"] == "label_pasal"].copy()

df_r["source"] = "Retrieval"
df_p["source"] = "Prediction"

df_all = pd.concat([
    df_r[["model", "accuracy", "f1_score", "source"]],
    df_p[["model", "accuracy", "f1_score", "source"]],
], ignore_index=True)

models = df_all["model"].tolist()
acc    = df_all["accuracy"].tolist()
f1     = df_all["f1_score"].tolist()
x      = np.arange(len(models))
width  = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, acc, width, label="Accuracy", color="#4C72B0", alpha=0.85)
bars2 = ax.bar(x + width/2, f1,  width, label="F1-Score",  color="#DD8452", alpha=0.85)

ax.set_title("Perbandingan Accuracy & F1-Score — label_pasal (362 vs 363)", fontsize=13)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.15)
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=35, ha="right", fontsize=8)
ax.legend(fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.5)

for bar in bars1:
    h = bar.get_height()
    ax.annotate(f"{h:.2f}", xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)
for bar in bars2:
    h = bar.get_height()
    ax.annotate(f"{h:.2f}", xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)

plt.tight_layout()
chart_path = EVAL_DIR / "evaluation_chart.png"
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart disimpan: {chart_path}")

## 5.6 Visualisasi MAE vonis_bulan

In [ ]:
if not df_vonis_mae.empty:
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    bars = ax2.barh(df_vonis_mae["model"], df_vonis_mae["mae_bulan"],
                    color="#55A868", alpha=0.85)
    ax2.set_xlabel("MAE (bulan)")
    ax2.set_title("Mean Absolute Error Prediksi Vonis (bulan)", fontsize=12)
    ax2.grid(axis="x", linestyle="--", alpha=0.5)
    for bar in bars:
        w = bar.get_width()
        ax2.annotate(f"{w:.2f} bln", xy=(w, bar.get_y() + bar.get_height()/2),
                     xytext=(3, 0), textcoords="offset points", va="center", fontsize=9)
    plt.tight_layout()
    plt.savefig(EVAL_DIR / "mae_vonis_chart.png", dpi=150, bbox_inches="tight")
    plt.show()

## 5.7 Error Analysis

In [ ]:
for model_name, col in pred_cols.items():
    if col not in df_pred.columns:
        continue
    errors = df_pred[df_pred[col] != df_pred["ground_truth_pasal"]]
    if not errors.empty:
        print(f"\n[{model_name}] Kesalahan Prediksi label_pasal:")
        for _, row in errors.iterrows():
            print(f"  {row['query_id']}: GT={row['ground_truth_pasal']} -> Pred={row[col]}")
    else:
        print(f"\n[{model_name}] Tidak ada kesalahan prediksi label_pasal ✓")

## 5.8 Analisis dan Rekomendasi

In [ ]:
best_pasal = df_pred_metrics.loc[df_pred_metrics["f1_score"].idxmax()]
print(f"Model Terbaik (label_pasal):")
print(f"  {best_pasal['model']} — Accuracy={best_pasal['accuracy']:.4f}, F1={best_pasal['f1_score']:.4f}")

if not df_vonis_mae.empty:
    best_vonis = df_vonis_mae.loc[df_vonis_mae["mae_bulan"].idxmin()]
    print(f"\nModel Terbaik (vonis_bulan MAE terendah):")
    print(f"  {best_vonis['model']} — MAE={best_vonis['mae_bulan']:.2f} bulan")

print("""
Temuan & Rekomendasi:
1. TF-IDF + SVM unggul untuk klasifikasi label_pasal karena teks hukum
   mengandung keyword spesifik yang mudah ditangkap oleh fitur TF-IDF.
2. Weighted Vote (similarity^5) lebih akurat daripada Majority Vote
   karena kasus paling mirip langsung mendominasi hasil prediksi.
3. Prediksi vonis_bulan (integer) lebih informatif daripada kategori
   ringan/sedang/berat dan memungkinkan pengukuran MAE yang lebih presisi.
4. IndoBERT memerlukan fine-tuning pada domain hukum Indonesia
   untuk mencapai performa optimal.
""")